# Train MedSAM — Attribute Segmentation
Runs on **Kaggle**. Trains the SAM encoder + custom multi-label decoder on ISIC 2018 Task 2
to segment 5 dermoscopic attributes: `globules`, `milia_like_cyst`, `negative_network`,
`pigment_network`, and `streaks`.

---
## 0. Environment setup

In [ ]:
!nvidia-smi
!pip install -q segment-anything scikit-learn tqdm

In [ ]:
# Download SAM ViT-B checkpoint
import os, subprocess

ckpt_path = "/kaggle/working/medsam_vit_b.pth"
if not os.path.exists(ckpt_path):
    subprocess.run(
        ["wget", "-q",
         "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
         "-O", ckpt_path],
        check=True,
    )
    print(f"Downloaded → {ckpt_path}")
else:
    print(f"Checkpoint already exists: {ckpt_path}")

In [ ]:
# Copy src and attribute dataset into /kaggle/working
import shutil, sys
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input/datasets/nguyenquynghia")
WORKING_DIR  = Path("/kaggle/working")

# src
src_working = WORKING_DIR / "src"
if src_working.exists():
    shutil.rmtree(src_working)
shutil.copytree(KAGGLE_INPUT / "src-training/src", src_working)
if str(src_working) not in sys.path:
    sys.path.insert(0, str(src_working))
print(f"src → {src_working}")

# Attribute prepared dataset  (dataset/attributes)
dataset_dest = WORKING_DIR / "dataset" / "attributes"
if dataset_dest.exists():
    shutil.rmtree(dataset_dest)
dataset_dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(KAGGLE_INPUT / "isic-2018-prepared/dataset/attributes", dataset_dest)
print(f"dataset/attributes → {dataset_dest}")

---
## 1. Train attribute segmentation

In [ ]:
from medsam.train_attributes import train as train_attr

train_attr(epochs=80, batch=4, lr=1e-4)

---
## 2. Save weights

In [ ]:
import zipfile
from pathlib import Path

output_dir = Path("/kaggle/working/outputs/attributes")
output_zip = "/kaggle/working/medsam_attributes_weights.zip"

if output_dir.exists():
    with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in output_dir.rglob("*"):
            if f.is_file():
                zf.write(f, f.relative_to("/kaggle/working"))
                print(f"  {f.relative_to('/kaggle/working')}")
    print(f"\nSaved → {output_zip}  (download from Kaggle Output tab)")
else:
    print("No attribute output directory found.")